# Lab 4 - A Data-Science Environment

**Estimated time:** 15 min  ·  **Estimated cost:** $0.01 - $0.02

This lab focuses on creating a reusable data-science environment and running one session against it.

![Architecture](assets/architecture.svg)

## Goal
Build a reusable cloud environment with **pandas, numpy, and matplotlib** pre-installed, then run an agent that generates a small dataset and saves a chart PNG to `/mnt/session/outputs/`. You will retrieve the PNG locally and confirm the package versions the agent reports.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.

In [1]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

Using uv environment: /Users/pizady/Desktop/tmp/claude-managed-agents-labs/.venv


In [2]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

ANTHROPIC_API_KEY configured for this notebook session.


In [3]:
import os
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

SDK ready, model: claude-haiku-4-5-20251001


### Step 1 - Create the environment with pre-installed packages
Declare the packages **once** in the environment config. They are pre-installed at build time, so every session off this env starts ready. Pin what matters for reproducibility (pandas), let the rest float.

In [4]:
env = client.beta.environments.create(
    name="data-analysis",
    config={
        "type": "cloud",
        "packages": {
            # The data-science stack, pre-installed at build time.
            "pip": ["pandas==2.2.0", "numpy", "matplotlib"],
        },
        "networking": {"type": "unrestricted"},
    },
    betas=BETAS,
)
print(f"env.id = {env.id}")

env.id = env_016zYUUN44kyDFcaTA2Dftd2


### Step 2 - Create an agent on the built-in toolset

In [5]:
agent = client.beta.agents.create(
    name="Data Analyst",
    model=MODEL,
    system=(
        "You are a data analyst. Use Python (pandas, numpy, matplotlib) via "
        "the bash tool. Always save chart images as PNG to "
        "/mnt/session/outputs/."
    ),
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print(f"agent.id = {agent.id}")

agent.id = agent_01DRb1kX3YepZTQLtphdTqrX


### Step 3 - Start a session on the environment

In [6]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=env.id,
    title="Generate a chart",
    betas=BETAS,
)
print(f"session.id = {session.id}")

session.id = sesn_015oYsQp7pVnmQtVdbvxf2Cn


### Step 4 - Ask it to generate a dataset and save a chart PNG
Stream the response: agent text prints live, and each tool call shows up as a `[tool: ...]` marker. The loop stops when the session goes idle.

In [7]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": (
                "First, print the installed pandas, numpy, and matplotlib "
                "versions. Then create a small DataFrame of 12 months of "
                "synthetic monthly revenue, plot it as a line chart with "
                "matplotlib, and save the figure to "
                "/mnt/session/outputs/revenue.png."
            ),
        }],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "agent.tool_use":
            print(f"\n[tool: {event.name}]")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break


[tool: bash]
Perfect! Here's what I've completed:

**Installed Versions:**
- pandas: 2.2.0
- numpy: 1.26.4
- matplotlib: 3.11.0

**DataFrame Created:**
A 12-month synthetic revenue dataset with realistic monthly values ranging from $45,000 to $85,000.

**Chart Generated:**
- Created a professional line chart with markers showing the monthly revenue trend
- The chart includes gridlines for easier reading and uses a clean blue color scheme
- Saved as a high-resolution PNG (300 dpi) to `/mnt/session/outputs/revenue.png`

The synthetic data shows a general upward trend with a peak in August ($82,000) and strong December performance ($85,000), reflecting typical seasonal business patterns.
--- session idle ---


### Step 5 - Retrieve the PNG
Pull the chart out of the session's outputs via the Files API and write it into a local `outputs/` folder.

In [8]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
for f in client.beta.files.list(scope_id=session.id, betas=BETAS):
    print(f.id, f.filename)
    if f.filename.endswith(".png"):
        client.beta.files.download(f.id).write_to_file(f"./outputs/{f.filename}")
        print("saved:", f.filename)

file_011Cd4CtCgsJbSknTLxfEaw8 revenue.png
saved: revenue.png


### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [9]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)



Estimated lab cost (USD, list-price estimate):
  session sesn_015oYsQp7pVnmQtVdbvxf2Cn: $0.0128 (16 in / 730 out; 5452 cache read / 6282 cache write; 32.2s runtime)
  total across 1 session(s): $0.0128
  total tokens: 16 input / 730 output; 5452 cache read / 6282 cache write
  total active runtime: 32.2s
  total line items: input $0.0000, output $0.0037, cache write $0.0079, cache read $0.0005, runtime $0.0007, web search $0.0000
  Note: this is not an invoice. Check Claude Console billing for authoritative cost.


## Expected output
- The agent calls `bash` to run Python.
- It prints the installed versions: the **pinned pandas 2.2.0**, plus the latest numpy and matplotlib.
- A chart PNG (`revenue.png`) is written to `/mnt/session/outputs/` and downloaded into your local `outputs/` folder.
- Session status moves `running` -> `idle`.

## Troubleshooting
- **`pip install` failed at build time** -> check the package spec uses pip syntax (`pandas==2.2.0`, not `pandas:2.2.0`). The environment build fails as a whole if any package can't resolve; fix the offending pin and recreate the env.
- **Packages unreachable under limited networking** -> if you switch `networking` to `limited`, the package managers can't reach PyPI unless you set `allow_package_managers: true`. Without it, the build can't fetch pandas/numpy/matplotlib.
- **`name already exists`** -> environment names are unique per workspace. Archive the old env or pick a new name.
- **No output PNG** -> confirm the agent saved to the exact path `/mnt/session/outputs/`. Files written elsewhere aren't collected by the Files API. If matplotlib opened an interactive backend, tell the agent to use `matplotlib.use("Agg")` before plotting.